# Unidad 3: Búsqueda, Optimización y Agentes Inteligentes
## 🔗 Problemas de Satisfacción de Restricciones (CSP)
### Inteligencia Artificial — Lic. en Sistemas — FCAD/UNER

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/ia-ls-fcad-uner/blob/main/notebooks/search/06_Problemas_Satisfaccion_Restricciones.ipynb)

---

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook vas a poder:

✅ Comprender qué es un **Problema de Satisfacción de Restricciones (CSP)** y sus componentes  
✅ Formular problemas reales como CSPs (mapas, reinas, sudoku, criptaritmo)  
✅ Resolver CSPs con **`python-constraint`** — la más didáctica  
✅ Resolver CSPs con **`OR-Tools`** — la más potente  
✅ Comparar enfoques y entender cuándo usar cada herramienta  
✅ Aplicar **propagación de restricciones** y **backtracking** en la práctica

---

## 📖 Marco Teórico

### ¿Qué es un CSP?

Un **Constraint Satisfaction Problem (CSP)** es un problema que se define mediante tres componentes:

| Componente | Descripción | Ejemplo (Coloreo de Mapa) |
|-----------|-------------|------------------------|
| **Variables (X)** | Entidades a asignar | Regiones: {NSW, QLD, WA, SA, VIC, TAS} |
| **Dominios (D)** | Valores posibles de cada variable | Colores: {Rojo, Verde, Azul} para cada región |
| **Restricciones (C)** | Condiciones que deben cumplirse | Regiones adyacentes ≠ mismo color |

**Objetivo**: Encontrar una **asignación consistente y completa** — valores para todas las variables que satisfacen todas las restricciones.

### Conceptos clave

- **Asignación consistente**: No viola ninguna restricción
- **Asignación completa**: Todas las variables tienen un valor
- **Solución**: Asignación consistente Y completa
- **Propagación de restricciones**: Reduce el dominio de variables usando restricciones (e.g., AC-3)
- **Backtracking**: Búsqueda en profundidad que revierte asignaciones cuando falla

### Características de los CSPs

✅ **Estructura factorizada** — el estado es un conjunto de variables, no una "caja negra"  
✅ **Heurísticas generales** — MRV (Minimum Remaining Values), grado, verificación hacia adelante  
✅ **Eficiencia** — la propagación de restricciones poda enormemente el espacio de búsqueda  
✅ **Aplicabilidad** — coloreo de mapas, sudoku, horarios, asignación de recursos

### Problemas clásicos de CSP

1. **Coloreo de Mapas**: Asignar colores a regiones sin que adyacentes compartan color
2. **N-Reinas**: Colocar N reinas en un tablero sin que se ataquen
3. **Sudoku**: Llenar un grid 9×9 con dígitos 1-9 respetando restricciones Alldiff
4. **Criptaritmo**: Asignar dígitos a letras en ecuaciones (ej: SEND + MORE = MONEY)
5. **Horarios**: Asignar cursos a aulas y docentes minimizando conflictos
6. **Asignación de recursos**: Distribuir recursos bajo restricciones de capacidad

---

## 📦 Paso 1: Instalación de librerías

Instalaremos las principales librerías para resolver CSPs en Python.

In [ ]:
# Instalar librerías necesarias
!pip install python-constraint ortools --quiet

print("✅ Librerías instaladas correctamente")

In [ ]:
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations, permutations
from constraint import Problem
from ortools.sat.python import cp_model
import time

# Función helper para la restricción "todos diferentes"
def all_different(*args):
    """Retorna True si todos los argumentos son distintos."""
    return len(set(args)) == len(args)

print("✅ Imports completados")


## 🔧 Paso 2: CSP — Conceptos básicos con un ejemplo simple

Empecemos con un CSP trivial:
- Variables: A, B, C
- Dominios: cada una toma valores {1, 2, 3}
- Restricciones: A ≠ B, B ≠ C, A ≠ C

In [ ]:
# Ejemplo simple: A ≠ B, B ≠ C, A ≠ C
# Esto es: todas las variables deben ser diferentes

print("CSP simple:")
print("  Variables: A, B, C")
print("  Dominios:  {1, 2, 3}")
print("  Restricciones: A ≠ B, B ≠ C, A ≠ C (Alldiff)")
print("\n¿Cuántas soluciones posibles hay?")

# Fuerza bruta: todas las asignaciones
soluciones = []
for a in [1, 2, 3]:
    for b in [1, 2, 3]:
        for c in [1, 2, 3]:
            if a != b and b != c and a != c:
                soluciones.append((a, b, c))

print(f"Total: {len(soluciones)} soluciones")
print("\nPrimeras 5 soluciones:")
for sol in soluciones[:5]:
    print(f"  A={sol[0]}, B={sol[1]}, C={sol[2]}")

## 📚 Paso 3: `python-constraint` — La más didáctica

**Ventajas**: Fácil de aprender, ideal para enseñanza, permite ver cómo funciona backtracking.  
**Desventajas**: No escala bien a problemas grandes.

### 3.1 — Ejemplo simple con python-constraint

In [ ]:
# Resolver el CSP simple con python-constraint
problem = Problem()

# Agregar variables y dominios
problem.addVariable("A", [1, 2, 3])
problem.addVariable("B", [1, 2, 3])
problem.addVariable("C", [1, 2, 3])

# Agregar restricciones
problem.addConstraint(lambda a, b: a != b, ("A", "B"))
problem.addConstraint(lambda b, c: b != c, ("B", "C"))
problem.addConstraint(lambda a, c: a != c, ("A", "C"))

# Resolver
soluciones = problem.getSolutions()

print(f"✅ Soluciones encontradas: {len(soluciones)}")
print("\nPrimeras 5:")
for sol in soluciones[:5]:
    print(f"  {sol}")

### 3.2 — Coloreo de mapas (Australia)

In [ ]:
# Problema: colorear el mapa de Australia con 3 colores
# Regiones: NSW, QLD, SA, WA, VIC, TAS
# Restricciones: regiones adyacentes no pueden tener el mismo color

# Mapa de Australia: {región: [vecinos]}
AUSTRALIA_MAP = {
    'WA': ['SA', 'NT'],
    'NT': ['WA', 'QLD', 'SA'],
    'SA': ['WA', 'NT', 'QLD', 'NSW', 'VIC'],
    'QLD': ['NT', 'SA', 'NSW'],
    'NSW': ['QLD', 'SA', 'VIC'],
    'VIC': ['SA', 'NSW', 'TAS'],
    'TAS': ['VIC'],
}

COLORES = ['Rojo', 'Verde', 'Azul']

# Crear el CSP
problema = Problem()

# Variables: cada región
for region in AUSTRALIA_MAP.keys():
    problema.addVariable(region, COLORES)

# Restricciones: regiones adyacentes ≠ color
for region, vecinos in AUSTRALIA_MAP.items():
    for vecino in vecinos:
        problema.addConstraint(lambda r, v: r != v, (region, vecino))

# Resolver
soluciones_mapa = problema.getSolutions()

print(f"🗺️ Coloreo de Australia")
print(f"   Regiones: {len(AUSTRALIA_MAP)}")
print(f"   Colores: {len(COLORES)}")
print(f"   Soluciones encontradas: {len(soluciones_mapa)}\n")

# Mostrar primera solución
if soluciones_mapa:
    sol = soluciones_mapa[0]
    print("Primera solución:")
    for region, color in sorted(sol.items()):
        print(f"  {region:5s} → {color}")

### 3.3 — El problema de las N-Reinas

In [ ]:
def resolver_n_reinas_constraint(n=8):
    """
    Resuelve el problema de las N-Reinas usando python-constraint.
    Variables: Xᵢ = columna de la reina en la fila i
    Dominio: {0, 1, ..., n-1}
    Restricciones:
      - Todas las columnas diferentes (Alldiff)
      - Diagonales: |fila_i - fila_j| ≠ |col_i - col_j|
    """
    problema = Problem()
    # Variables: posición de la reina en cada fila
    for i in range(n):
        problema.addVariable(f"R{i}", list(range(n)))
    # Restricción 1: todas las reinas en columnas diferentes (Alldiff)
    variables = [f"R{i}" for i in range(n)]
    problema.addConstraint(all_different, variables)
    # Restricción 2: no se atacan en diagonales
    for i in range(n):
        for j in range(i + 1, n):
            # |fila_i - fila_j| ≠ |col_i - col_j|
            # Ya tenemos que i ≠ j (filas distintas)
            # Falta que i - j ≠ col_i - col_j (diagonal principal)
            #             i - j ≠ col_j - col_i (diagonal secundaria)
            problema.addConstraint(
                lambda c_i, c_j, row_i=i, row_j=j: abs(row_i - row_j) != abs(c_i - c_j),
                (f"R{i}", f"R{j}")
            )
    return problema.getSolutions()
# Resolver para 8-Reinas
t0 = time.time()
soluciones_8reinas = resolver_n_reinas_constraint(8)
tiempo = time.time() - t0
print(f"♟️ Problema de las 8-Reinas (python-constraint)")
print(f"   Soluciones encontradas: {len(soluciones_8reinas)}")
print(f"   Tiempo: {tiempo:.4f}s\n")
if soluciones_8reinas:
    sol = soluciones_8reinas[0]
    print("Primera solución (columnas por fila):")
    posiciones = [sol[f"R{i}"] for i in range(8)]
    print(f"  {posiciones}")
    # Visualizar
    tablero = np.zeros((8, 8))
    for fila, col in enumerate(posiciones):
        tablero[fila, col] = 1
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(tablero, cmap='Greys', vmin=0, vmax=1)
    # Dibujar reinas
    for fila, col in enumerate(posiciones):
        ax.text(col, fila, '♛', ha='center', va='center',
               fontsize=30, color='red')
    # Cuadrícula
    ax.set_xticks(np.arange(-0.5, 8, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 8, 1), minor=True)
    ax.grid(which='minor', color='black', linewidth=2)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title('8-Reinas — Primera Solución', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


## 🚀 Paso 4: OR-Tools — La más potente

**Ventajas**: Muy eficiente, optimización avanzada, scheduling, industrial.  
**Desventajas**: Más compleja para principiantes.

### 4.1 — N-Reinas con OR-Tools (más eficiente)

In [ ]:
def resolver_n_reinas_ortools(n=8):
    """
    Resuelve N-Reinas con OR-Tools CP-SAT.
    Mucho más eficiente que python-constraint.
    """
    model = cp_model.CpModel()

    # Variables: posición de reina en cada fila
    queens = {}
    for i in range(n):
        queens[i] = model.NewIntVar(0, n - 1, f'queen_{i}')

    # Restricción 1: todas en columnas distintas
    model.AddAllDifferent(list(queens.values()))

    # Restricción 2: diagonales
    for i in range(n):
        for j in range(i + 1, n):
            # |fila_i - fila_j| ≠ |col_i - col_j|
            # Como las filas son distintas, solo chequeamos diagonales
            model.Add(queens[i] - queens[j] != i - j)  # diagonal principal
            model.Add(queens[i] - queens[j] != j - i)  # diagonal secundaria

    # Resolver
    solver = cp_model.CpSolver()
    solver.parameters.linearization_level = 0
    
    # Recopilar TODAS las soluciones (callback)
    soluciones = []
    
    class SolutionCollector(cp_model.CpSolverSolutionCallback):
        def on_solution_callback(self):
            sol = {i: self.Value(queens[i]) for i in range(n)}
            soluciones.append(sol)
    
    solver.Solve(model, SolutionCollector())
    
    return soluciones


# Comparar tiempos
print("Comparación: python-constraint vs OR-Tools\n")

t0 = time.time()
sols_constraint = resolver_n_reinas_constraint(8)
t_constraint = time.time() - t0

t0 = time.time()
sols_ortools = resolver_n_reinas_ortools(8)
t_ortools = time.time() - t0

print(f"python-constraint:")
print(f"  Soluciones: {len(sols_constraint)}")
print(f"  Tiempo: {t_constraint:.4f}s\n")

print(f"OR-Tools (CP-SAT):")
print(f"  Soluciones: {len(sols_ortools)}")
print(f"  Tiempo: {t_ortools:.4f}s\n")

speedup = t_constraint / t_ortools if t_ortools > 0 else float('inf')
print(f"Speedup: {speedup:.1f}x más rápido con OR-Tools")

### 4.2 — Sudoku con OR-Tools

In [ ]:
def resolver_sudoku_ortools(grid):
    """
    Resuelve un Sudoku usando OR-Tools.
    grid: matriz 9x9 con 0 en celdas vacías
    """
    model = cp_model.CpModel()

    # Variables: cada celda puede ser 1-9
    x = {}
    for i in range(9):
        for j in range(9):
            if grid[i][j] == 0:
                x[i, j] = model.NewIntVar(1, 9, f'cell_{i}_{j}')
            else:
                x[i, j] = grid[i][j]  # Celdas fijas

    # Restricción 1: todas diferentes en cada fila
    for i in range(9):
        fila = [x[i, j] for j in range(9)]
        model.AddAllDifferent(fila)

    # Restricción 2: todas diferentes en cada columna
    for j in range(9):
        columna = [x[i, j] for i in range(9)]
        model.AddAllDifferent(columna)

    # Restricción 3: todas diferentes en cada caja 3x3
    for box_i in range(3):
        for box_j in range(3):
            caja = []
            for i in range(3 * box_i, 3 * box_i + 3):
                for j in range(3 * box_j, 3 * box_j + 3):
                    caja.append(x[i, j])
            model.AddAllDifferent(caja)

    # Resolver
    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    # Reconstruir solución
    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        solucion = np.zeros((9, 9), dtype=int)
        for i in range(9):
            for j in range(9):
                solucion[i, j] = solver.Value(x[i, j]) if isinstance(x[i, j], cp_model.IntVar) else x[i, j]
        return solucion
    return None


# Ejemplo: Sudoku fácil
sudoku_facil = np.array([
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9]
])

print("🧩 Sudoku — Entrada:")
print(sudoku_facil)

t0 = time.time()
solucion = resolver_sudoku_ortools(sudoku_facil)
tiempo = time.time() - t0

print(f"\n✅ Sudoku — Solución (en {tiempo:.4f}s):")
print(solucion)

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, grid, titulo in [(axes[0], sudoku_facil, 'Entrada'),
                          (axes[1], solucion, 'Solución')]:
    ax.imshow(np.ones((9, 9)), cmap='Greys', vmin=0, vmax=1)
    
    # Números
    for i in range(9):
        for j in range(9):
            val = grid[i, j]
            if val > 0:
                # Color diferente si es número original
                color = 'black' if sudoku_facil[i, j] > 0 else 'blue'
                ax.text(j, i, str(int(val)), ha='center', va='center',
                       fontsize=14, fontweight='bold', color=color)
    
    # Cuadrículas
    for i in range(10):
        if i % 3 == 0:
            ax.axhline(i - 0.5, color='black', linewidth=2)
            ax.axvline(i - 0.5, color='black', linewidth=2)
        else:
            ax.axhline(i - 0.5, color='gray', linewidth=0.5)
            ax.axvline(i - 0.5, color='gray', linewidth=0.5)
    
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(titulo, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 🔐 Paso 5: Criptaritmo — SEND + MORE = MONEY

Clásico problema de CSP: Asignar dígitos (0-9) a letras de modo que la operación aritmética sea válida.

In [ ]:
def resolver_criptaritmo_constraint():
    """
    SEND + MORE = MONEY
    S, E, N, D, M, O, R, Y son dígitos 0-9, todos diferentes.
    S, M ≠ 0 (primera letra no puede ser 0)
    """
    problema = Problem()

    letras = ['S', 'E', 'N', 'D', 'M', 'O', 'R', 'Y']
    
    # Dominios
    for letra in letras:
        problema.addVariable(letra, list(range(10)))

    # Restricción 1: todas diferentes
    problema.addConstraint(AllDifferent(letras))

    # Restricción 2: primera letra ≠ 0
    problema.addConstraint(lambda s: s != 0, ('S',))
    problema.addConstraint(lambda m: m != 0, ('M',))

    # Restricción 3: la ecuación SEND + MORE = MONEY
    def ecuacion(s, e, n, d, m, o, r, y):
        SEND = 1000*s + 100*e + 10*n + d
        MORE = 1000*m + 100*o + 10*r + e
        MONEY = 10000*m + 1000*o + 100*n + 10*e + y
        return SEND + MORE == MONEY

    problema.addConstraint(ecuacion, letras)

    return problema.getSolutions()


def resolver_criptaritmo_ortools():
    """
    Mismo problema con OR-Tools.
    """
    model = cp_model.CpModel()

    # Variables
    S = model.NewIntVar(1, 9, 'S')   # S ≠ 0
    E = model.NewIntVar(0, 9, 'E')
    N = model.NewIntVar(0, 9, 'N')
    D = model.NewIntVar(0, 9, 'D')
    M = model.NewIntVar(1, 9, 'M')   # M ≠ 0
    O = model.NewIntVar(0, 9, 'O')
    R = model.NewIntVar(0, 9, 'R')
    Y = model.NewIntVar(0, 9, 'Y')

    # Todas diferentes
    model.AddAllDifferent([S, E, N, D, M, O, R, Y])

    # Ecuación: SEND + MORE = MONEY
    SEND = 1000*S + 100*E + 10*N + D
    MORE = 1000*M + 100*O + 10*R + E
    MONEY = 10000*M + 1000*O + 100*N + 10*E + Y

    model.Add(SEND + MORE == MONEY)

    # Resolver
    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        return {
            'S': solver.Value(S), 'E': solver.Value(E),
            'N': solver.Value(N), 'D': solver.Value(D),
            'M': solver.Value(M), 'O': solver.Value(O),
            'R': solver.Value(R), 'Y': solver.Value(Y),
        }
    return None


print("🔐 Criptaritmo: SEND + MORE = MONEY\n")

# Resolver con python-constraint
t0 = time.time()
sols_constraint = resolver_criptaritmo_constraint()
t_constraint = time.time() - t0

# Resolver con OR-Tools
t0 = time.time()
sol_ortools = resolver_criptaritmo_ortools()
t_ortools = time.time() - t0

print("python-constraint:")
print(f"  Soluciones: {len(sols_constraint)}")
print(f"  Tiempo: {t_constraint:.4f}s")
if sols_constraint:
    sol = sols_constraint[0]
    s = sol['S']; e = sol['E']; n = sol['N']; d = sol['D']
    m = sol['M']; o = sol['O']; r = sol['R']; y = sol['Y']
    print(f"  Solución: S={s} E={e} N={n} D={d} M={m} O={o} R={r} Y={y}")
    SEND = 1000*s + 100*e + 10*n + d
    MORE = 1000*m + 100*o + 10*r + e
    MONEY = 10000*m + 1000*o + 100*n + 10*e + y
    print(f"  Verificación: {SEND} + {MORE} = {MONEY} ✓")

print(f"\nOR-Tools:")
print(f"  Tiempo: {t_ortools:.4f}s")
if sol_ortools:
    s = sol_ortools['S']; e = sol_ortools['E']; n = sol_ortools['N']; d = sol_ortools['D']
    m = sol_ortools['M']; o = sol_ortools['O']; r = sol_ortools['R']; y = sol_ortools['Y']
    print(f"  Solución: S={s} E={e} N={n} D={d} M={m} O={o} R={r} Y={y}")
    SEND = 1000*s + 100*e + 10*n + d
    MORE = 1000*m + 100*o + 10*r + e
    MONEY = 10000*m + 1000*o + 100*n + 10*e + y
    print(f"  Verificación: {SEND} + {MORE} = {MONEY} ✓")

## 📊 Paso 6: Comparativa de librerías

Resumen de cuándo usar cada herramienta.

In [ ]:
import pandas as pd

comparativa = pd.DataFrame([
    {
        'Librería': 'python-constraint',
        'Curva de aprendizaje': '⭐ Muy fácil',
        'Eficiencia': '⭐ Baja',
        'Escalabilidad': '❌ Pequeños problemas',
        'Ideal para': 'Enseñanza y prototipos',
        'Ejemplo': 'N-Reinas, Mapas',
    },
    {
        'Librería': 'OR-Tools (CP-SAT)',
        'Curva de aprendizaje': '⭐⭐⭐ Intermedia',
        'Eficiencia': '⭐⭐⭐⭐⭐ Muy alta',
        'Escalabilidad': '✅ Grandes problemas',
        'Ideal para': 'Industria, optimización',
        'Ejemplo': 'Scheduling, Routing',
    },
    {
        'Librería': 'PyCSP3',
        'Curva de aprendizaje': '⭐⭐ Fácil',
        'Eficiencia': '⭐⭐⭐⭐ Alta',
        'Escalabilidad': '✅ Problemas medianos',
        'Ideal para': 'Investigación',
        'Ejemplo': 'Problemas complejos',
    },
], columns=['Librería', 'Curva de aprendizaje', 'Eficiencia', 'Escalabilidad', 'Ideal para', 'Ejemplo'])

print("📊 Comparativa de librerías CSP\n")
print(comparativa.to_string(index=False))

print("\n" + "="*80)
print("RECOMENDACIÓN PARA ENSEÑANZA:")
print("="*80)
print("\n1️⃣ Primer contacto → python-constraint")
print("   - Entender: variables, dominios, restricciones, backtracking")
print("   - Sin distraerse con detalles técnicos")
print("\n2️⃣ Nivel intermedio → OR-Tools")
print("   - Propagación eficiente")
print("   - Problemas más complejos")
print("   - Optimización")
print("\n3️⃣ Nivel avanzado → PyCSP3 + OR-Tools")
print("   - Investigación")
print("   - Problemas especializados")

## 🎓 Resumen y Conclusiones

### Puntos clave

1. **Los CSPs son una abstracción poderosa** para problemas combinatorios: variables, dominios, restricciones.

2. **Tres técnicas esenciales**:
   - **Propagación de restricciones** (AC-3): reduce dominios antes de buscar
   - **Backtracking**: búsqueda sistemática con retroceso
   - **Heurísticas** (MRV, Degree): guían la búsqueda inteligentemente

3. **Problemas clásicos** muestran aplicabilidad:
   - Coloreo de mapas: planificación territorial
   - N-Reinas: sistemas de recursos
   - Sudoku: inferencia y lógica
   - Criptaritmo: ecuaciones con restricciones

4. **Librerías Python** hacen los CSPs accesibles:
   - **python-constraint**: didáctica, fácil de aprender
   - **OR-Tools**: industrial, eficiente, optimización
   - **PyCSP3**: investigación, modelado avanzado

5. **Estructura importa**: Un grafo de restricciones con estructura (árbol, bajo treewidth) es mucho más fácil de resolver.

### 🚀 ¿Qué sigue?

- **Problemas de optimización**: minimizar costo sujeto a restricciones (COP)
- **Scheduling avanzado**: asignación de recursos, horarios, turnos
- **Routing y logística**: problema del viajante (TSP), rutas de vehículos (VRP)
- **CSPs distribuidos**: múltiples agentes resolviendo en paralelo
- **Machine Learning + CSP**: integración con aprendizaje automático

### 📚 Referencias

- Russell, S. & Norvig, P. (2020). *Artificial Intelligence: A Modern Approach* (4th ed.). **Capítulo 6: Constraint Satisfaction Problems**
- [python-constraint documentation](https://python-constraint.github.io/)
- [OR-Tools official guide](https://developers.google.com/optimization)
- [PyCSP3 documentation](https://pycsp.org/)

---

*© 2026 Cátedra Inteligencia Artificial — Lic. en Sistemas — FCAD/UNER*  
[![CC BY-SA 4.0](https://licensebuttons.net/l/by-sa/4.0/88x31.png)](https://creativecommons.org/licenses/by-sa/4.0/)